# Análisis Ideológico Global e Índice ISID 2026

Este cuaderno reproduce los cálculos del **Índice Sintético de Orientación de Derechas (ISID)**, verifica las correlaciones entre los seis ejes ideológicos y exporta gráficos interactivos HTML en la carpeta `../charts/`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

DATA_PATH = Path('../data/ideologia_global_2026.csv')
CHARTS_DIR = Path('../charts')
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f'Países cargados: {len(df)}')
df.head()

Países cargados: 34


,country_en,pais,region,heritage_2026,fraser_2026,fraser_tamano_estado,hofstede_idv_2023,giem_wvs,gallup_ley_orden_2025,eiu_democracia_2025,...,eje_cultural_iw,pilar1_libre_mercado,pilar2_resp_individual,pilar3_ley_orden,isid_base,isid_mercado,isid_ley_orden,isid_giem,cumple_filtro_antiestatista,ranking_isid_filtrado
0,Switzerland,Suiza,Europa,83.7,8.45,7.67,79.0,0.573,91,9.32,...,Secular-Racional | Autoexpresión,81.88,77.85,91.0,83.58,82.49,86.09,81.20,True,1.0
1,Singapore,Singapur,Asia-Pacífico,84.4,8.43,7.38,43.0,NaN,95,6.18,...,Confuciano | Restricción/Orden,81.18,58.40,95.0,78.19,77.11,84.23,NaN,True,2.0
2,United States,Estados Unidos,Norteamérica,72.8,8.21,7.39,60.0,0.704,86,7.65,...,Más Tradicional | Autoexpresión,76.38,66.95,86.0,76.44,75.48,79.79,76.73,True,3.0
3,Lithuania,Lituania,Europa,75.0,7.82,6.75,55.0,NaN,91,7.55,...,Secular-Racional | Supervivencia-Mixto,73.87,61.25,91.0,75.37,73.51,80.77,NaN,True,4.0
4,Taiwan,Taiwán,Asia-Pacífico,79.8,7.90,7.57,40.0,0.591,89,8.78,...,Secular-Confuciano | Seguridad,78.29,57.85,89.0,75.05,74.30,80.09,74.89,True,5.0


In [2]:
# Recálculo transparente de pilares y pruebas de sensibilidad
df['p1_calc'] = 0.35 * df['heritage_2026'] + 0.35 * (df['fraser_2026'] * 10) + 0.30 * (df['fraser_tamano_estado'] * 10)
df['p2_calc'] = 0.50 * df['hofstede_idv_2023'] + 0.50 * (df['fraser_tamano_estado'] * 10)
df['p3_calc'] = df['gallup_ley_orden_2025']

df['isid_calc'] = (df['p1_calc'] + df['p2_calc'] + df['p3_calc']) / 3.0
df['isid_mercado_calc'] = 0.50 * df['p1_calc'] + 0.30 * df['p2_calc'] + 0.20 * df['p3_calc']
df['isid_orden_calc'] = 0.25 * df['p1_calc'] + 0.20 * df['p2_calc'] + 0.55 * df['p3_calc']

top10 = (
    df[(df['fraser_tamano_estado'] >= 6.0) & df['isid_calc'].notna()]
    .sort_values('isid_calc', ascending=False)
    .head(10)
    .copy()
)
top10[['pais', 'isid_calc', 'p1_calc', 'p2_calc', 'p3_calc', 'fraser_tamano_estado', 'eiu_democracia_2025']].round(2)

,pais,isid_calc,p1_calc,p2_calc,p3_calc,fraser_tamano_estado,eiu_democracia_2025
0,Suiza,83.58,81.88,77.85,91,7.67,9.32
1,Singapur,78.19,81.18,58.40,95,7.38,6.18
2,Estados Unidos,76.44,76.38,66.95,86,7.39,7.65
3,Lituania,75.37,73.87,61.25,91,6.75,7.55
4,Taiwán,75.05,78.29,57.85,89,7.57,8.78
5,Canadá,74.85,73.00,67.55,84,6.31,9.08
6,Estonia,74.66,73.97,62.00,88,6.20,8.07
7,Nueva Zelanda,74.43,75.44,65.85,82,6.27,9.62
8,República Checa,73.64,71.27,65.65,84,6.13,8.15
9,Australia,73.37,74.76,67.35,78,6.17,8.85


In [3]:
# Matriz de correlaciones entre ejes (muestra con datos de individualismo)
cols_corr = [
    'heritage_2026', 'fraser_2026', 'hofstede_idv_2023',
    'gallup_ley_orden_2025', 'eiu_democracia_2025', 'confianza_social_wvs'
]
corr_matrix = df.dropna(subset=['hofstede_idv_2023'])[cols_corr].corr().round(3)
corr_matrix

,heritage_2026,fraser_2026,hofstede_idv_2023,gallup_ley_orden_2025,eiu_democracia_2025,confianza_social_wvs
heritage_2026,1.000,0.857,0.434,0.401,0.637,0.367
fraser_2026,0.857,1.000,0.437,0.240,0.705,0.270
hofstede_idv_2023,0.434,0.437,1.000,-0.017,0.690,0.615
gallup_ley_orden_2025,0.401,0.240,-0.017,1.000,-0.079,0.290
eiu_democracia_2025,0.637,0.705,0.690,-0.079,1.000,0.283
confianza_social_wvs,0.367,0.270,0.615,0.290,0.283,1.000


In [4]:
# Gráfico 1: Libertad económica vs. Democracia (tamaño = Confianza social)
df_g1 = df.dropna(subset=['heritage_2026', 'eiu_democracia_2025']).copy()
df_g1['tamano_punto'] = df_g1['confianza_social_wvs'].fillna(30)

fig1 = px.scatter(
    df_g1,
    x='heritage_2026',
    y='eiu_democracia_2025',
    color='region',
    size='tamano_punto',
    text='pais',
    hover_data=['fraser_2026', 'gallup_ley_orden_2025', 'confianza_social_wvs'],
    title='Libertad Económica (Heritage 2026) vs. Democracia (EIU 2025)',
    labels={
        'heritage_2026': 'Libertad Económica Heritage 2026 (0-100)',
        'eiu_democracia_2025': 'Índice de Democracia EIU 2025 (0-10)',
        'region': 'Región'
    },
    template='plotly_white'
)
fig1.update_traces(textposition='top center', textfont_size=10)
fig1.add_vline(x=70, line_dash='dash', line_color='#888888')
fig1.add_hline(y=8.0, line_dash='dash', line_color='#888888')
fig1.write_html(CHARTS_DIR / '01_mercado_democracia_confianza.html')
fig1.show()

In [5]:
# Gráfico 2: Contención fiscal (Fraser Tamaño del Estado) vs. Individualismo (2023)
df_g2 = df.dropna(subset=['fraser_tamano_estado', 'hofstede_idv_2023']).copy()
df_g2['perfil'] = np.where(
    df_g2['fraser_tamano_estado'] < 6.0,
    'Individualismo Estatista (< 6.0)',
    np.where(df_g2['hofstede_idv_2023'] >= 60, 'Individualismo de Estado Limitado', 'Colectivismo Pro-Mercado')
)

fig2 = px.scatter(
    df_g2,
    x='fraser_tamano_estado',
    y='hofstede_idv_2023',
    color='perfil',
    text='pais',
    hover_data=['giem_wvs', 'isid_calc'],
    title='Contención Fiscal del Estado vs. Individualismo Cultural (2023)',
    labels={
        'fraser_tamano_estado': 'Fraser Área 1: Tamaño del Estado (mayor = Estado más limitado)',
        'hofstede_idv_2023': 'Individualismo Cultural 2023 (0-100)',
        'perfil': 'Cuadrante'
    },
    template='plotly_white'
)
fig2.update_traces(textposition='top center', marker_size=11)
fig2.add_vline(x=6.0, line_dash='dash', line_color='#666666', annotation_text='Umbral 6.0')
fig2.add_hline(y=60, line_dash='dash', line_color='#666666')
fig2.write_html(CHARTS_DIR / '02_individualismo_estado_giem.html')
fig2.show()

In [6]:
# Gráfico 3: Ranking Top 10 ISID y análisis de sensibilidad por ponderación
top10_plot = top10.iloc[::-1]

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    y=top10_plot['pais'],
    x=top10_plot['isid_calc'].round(2),
    orientation='h',
    name='ISID Equilibrado (33/33/33)',
    marker_color='#1A73E8',
    text=top10_plot['isid_calc'].round(2),
    textposition='outside'
))
fig3.add_trace(go.Scatter(
    y=top10_plot['pais'],
    x=top10_plot['isid_mercado_calc'].round(2),
    mode='markers',
    name='Énfasis Libre Mercado (50/30/20)',
    marker=dict(color='#12B5CB', size=10, symbol='diamond')
))
fig3.add_trace(go.Scatter(
    y=top10_plot['pais'],
    x=top10_plot['isid_orden_calc'].round(2),
    mode='markers',
    name='Énfasis Ley y Orden (25/20/55)',
    marker=dict(color='#FA903E', size=10, symbol='circle')
))

fig3.update_layout(
    title='Top 10 Países por Orientación de Derechas (ISID 2026) y Sensibilidad',
    xaxis=dict(title='Puntuación (0-100)', range=[65, 90]),
    yaxis=dict(title=''),
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig3.write_html(CHARTS_DIR / '03_ranking_isid_sensibilidad.html')
fig3.show()